# DLinear and Transformer Split-Horizon Forecasting, Leakage-Safe Selector (mixed4)

This notebook uses one shared BasicTS forecasting data pipeline for both models. DLinear forecasts steps 1-6, the Transformer forecasts steps 7-12, and their 6-step outputs are concatenated into one 12-step hybrid forecast.

The selector section avoids test-set leakage: hard selector masks and soft weights are learned from validation predictions, then applied once to the test predictions for final reporting.

Another thing before next meeting is that you should think what challenges existing paper has, so we have to propose current method. just add what you need from the past MOE Tell me the background first, explain that you think should explain, and make the results clear

## 1. Project Setup

This cell keeps the notebook runnable from inside `notebooks/` by moving to the repo root and adding `src/` to Python's import path.


In [ ]:
import os 
import sys
from pathlib import Path
#root contains the path to the basicts
ROOT = Path(r"C:\Users\luwil\OneDrive\Documents\Code\BasicTS")
#move python working folder
os.chdir(ROOT)
# the path to src is src_path
src_path = ROOT / "src"
#if src_path is not in the system path, add it to the system path
#system path is added to python search. 
# This allows us to import modules from the src 
# folder without having to specify the full path.
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

## 2. Imports and Shared Settings

Both models use the same dataset, scaler, preprocessing, `input_len`, train/val/test split, and batch format. The shared dataset keeps the full 12-step target window, while each split-horizon model trains on its own 6-step slice.


In [ ]:
import json
from datetime import datetime
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader

from basicts.configs import BasicTSForecastingConfig, BasicTSModelConfig
from basicts.launcher import BasicTSLauncher
from basicts.models.DLinear import DLinear, DLinearConfig
from basicts.models.iTransformer import iTransformerConfig, iTransformerForForecasting
from basicts.runners.builder import Builder
from basicts.runners.taskflow import BasicTSForecastingTaskFlow
from basicts.scaler import ZScoreScaler
from basicts.utils import BasicTSMode
DATASET_NAME = "ETTh1"
#most papers use 96 input length
INPUT_LEN = 96
#most papers use 96, 192, 336, and 720 input length
FULL_OUTPUT_LEN = 12

SPLIT_OUTPUT_LEN = 6
#etthl has 7 variables
NUM_FEATURES = 7
#Batch sizes like 16, 32, and 64 are normal. 32 is a safe default.
BATCH_SIZE = 32
# common for testing
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3
# Fresh namespace for this notebook so BasicTS does not auto-resume from old/corrupt checkpoints.
RUN_TAG = "mixed_4"
# this is the shared settings dictionary that both models use
SHARED_CONFIG = {
    "dataset_name": DATASET_NAME,
    "input_len": INPUT_LEN,
    "dataset_params": {
        "input_len": INPUT_LEN,
        "output_len": FULL_OUTPUT_LEN,
        "use_timestamps": False,
        "memmap": False,
    },
    "use_timestamps": False,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "scaler": ZScoreScaler,
    "norm_each_channel": True,
    "rescale": False,
    "metrics": ["MAE", "MSE"],
    "optimizer_params": {"lr": LEARNING_RATE, "weight_decay": 5e-4},
    "gpus": None,
    "train_data_num_workers": 0,
    "val_data_num_workers": 0,
    "test_data_num_workers": 0,
    "save_results": True,
}

# The standalone 12-step models only need BasicTS test_metrics.json.
# Metrics-only evaluation avoids Windows memmap file-lock issues.
FULL_12_STEP_CONFIG = dict(SHARED_CONFIG)
FULL_12_STEP_CONFIG["save_results"] = False


def fresh_checkpoint_dir(model_folder, run_name):
    # Each training launch gets a unique parent folder, so BasicTS cannot resume a stale checkpoint.
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return str(Path("checkpoints") / RUN_TAG / model_folder / run_name / stamp)

## 3. Shared Shape Check Helper

This helper builds the BasicTS dataset and scaler, runs the same forecasting preprocessing that training uses, and then sends one batch through the selected model.


In [ ]:
# for the split forecasitting model, we need to create a custom taskflow that slices the targets and target masks to the desired output length      
#start with the forecasting taskflwo and mofify it 
# BasicTSForecastingTaskFlow is the default data-prep worker.
# It prepares each forecasting batch before the model uses it.
# start with the deault taskflow and then add a change 
class SplitHorizonForecastingTaskFlow(BasicTSForecastingTaskFlow):
    #adding a setting called targest slide whchi is a variable that is used to slice the targets
    # tells the taskflow which targets to keep
    # need self because we need acresss to this specific object
    #creates a variables incide the class object 
    def __init__(self, target_slice):
        self.target_slice = target_slice
    # preprocess 
    def preprocess(self, runner, data):
        #normal work
        data = super().preprocess(runner, data)
        #cuts target values
        data["targets"] = data["targets"][:, self.target_slice, :]
        # tells basicts which target values are valid
        data["targets_mask"] = data["targets_mask"][:, self.target_slice, :]
        return data


def _float_batch(batch):
    return {
        key: value.float() if isinstance(value, torch.Tensor) and value.is_floating_point() else value
        for key, value in batch.items()
    }

# checks to see if the inputs are the right shape the targest are sliced and the prediction matches targer
def preview_shapes(cfg, model_name):
    #build the dataset using basicts
    train_dataset = Builder._build_dataset(cfg, BasicTSMode.TRAIN)
    # puts the dataset into batches gets ready for batches
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=False)
    #This grabs the first batch.
    raw_batch = _float_batch(next(iter(train_loader)))
    #creates the scaler and fits it to the training data  
    scaler = Builder._build_scaler(cfg)
    scaler.fit(train_dataset.data)
    # fake runner so the pasicts 
    class PreviewRunner:
        pass
    # Create a fake runner that has cfg and scaler, because taskflow.preprocess expects a runner object.
    runner = PreviewRunner() 
    runner.cfg = cfg
    runner.scaler = scaler
    #prepares the batch normally
    processed_batch = cfg.taskflow.preprocess(runner, dict(raw_batch))
    #build the model from the config and switch it to evaluation mode
    model = cfg.model(cfg.model_config)
    model.eval()
    # o not track gradient as they are only needed for trainig 
    with torch.no_grad():
        #sends processed inputs into the model
        prediction = model(processed_batch["inputs"])
        # then the model outputs future values
        # did the model return a dictionary
        if isinstance(prediction, dict):
            # if it did this extracts onlt the prediction tensor
            prediction = prediction["prediction"]
    #print the shapres to check that the data and model match before the training
    print(f"{model_name} raw inputs shape:       ", tuple(raw_batch["inputs"].shape))
    print(f"{model_name} raw target shape:       ", tuple(raw_batch["targets"].shape))
    print(f"{model_name} processed inputs shape: ", tuple(processed_batch["inputs"].shape))
    print(f"{model_name} target shape:           ", tuple(processed_batch["targets"].shape))
    print(f"{model_name} prediction shape:       ", tuple(prediction.shape))
    #checks
    assert tuple(raw_batch["targets"].shape) == (cfg.batch_size, FULL_OUTPUT_LEN, NUM_FEATURES)
    assert tuple(processed_batch["inputs"].shape) == (cfg.batch_size, INPUT_LEN, NUM_FEATURES)
    assert tuple(processed_batch["targets"].shape) == (cfg.batch_size, SPLIT_OUTPUT_LEN, NUM_FEATURES)
    assert tuple(prediction.shape) == (cfg.batch_size, SPLIT_OUTPUT_LEN, NUM_FEATURES)
    return processed_batch, prediction


## 4. DLinear Config

This config reuses `BasicTSForecastingConfig` and only changes the model-specific pieces. The shared dataset/scaler/preprocessing settings come from `SHARED_CONFIG`.


In [ ]:
# tells BasicTS how to build and train DLinear
dlinear_model_config = DLinearConfig(
    input_len=INPUT_LEN,
    output_len=SPLIT_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    moving_avg=25,
    stride=1,
    individual=False,
)

# full BasicTS training config for split-horizon DLinear
dlinear_cfg = BasicTSForecastingConfig(
    model=DLinear,
    model_config=dlinear_model_config,
    taskflow=SplitHorizonForecastingTaskFlow(slice(0, SPLIT_OUTPUT_LEN)),
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/DLinear/{DATASET_NAME}_{INPUT_LEN}_steps_1_6",
    **SHARED_CONFIG,
)

# Standalone DLinear trained to forecast all 12 steps.
dlinear_full_model_config = DLinearConfig(
    input_len=INPUT_LEN,
    output_len=FULL_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    moving_avg=25,
    stride=1,
    individual=False,
)

dlinear_full_cfg = BasicTSForecastingConfig(
    model=DLinear,
    model_config=dlinear_full_model_config,
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/DLinear/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **FULL_12_STEP_CONFIG,
)

dlinear_cfg, dlinear_full_cfg

## 5. DLinear Shape Test

Run this before training. The model prediction and processed target lines must be `(batch_size, 6, num_features)`, while the raw target line remains `(batch_size, 12, num_features)`.


In [ ]:
#checker
dlinear_batch, dlinear_prediction = preview_shapes(dlinear_cfg, "DLinear")


## 6. Train the DLinear

This is the first training run. Leave `RUN_DLinear_TRAINING` as `False` while editing or shape-checking, then switch it to `True` when you are ready to train.


In [ ]:
#training
RUN_DLinear_TRAINING = False
RUN_DLinear_12_STEP_TRAINING = False

if RUN_DLinear_TRAINING:
    dlinear_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "DLinear",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_6",
    )
    print("Training split DLinear in:", dlinear_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(dlinear_cfg)
else:
    print("DLinear split-horizon training skipped. Set RUN_DLinear_TRAINING = True to train.")

if RUN_DLinear_12_STEP_TRAINING:
    dlinear_full_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "DLinear",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    )
    print("Training 12-step DLinear in:", dlinear_full_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(dlinear_full_cfg)
else:
    print("DLinear 12-step training skipped. Set RUN_DLinear_12_STEP_TRAINING = True to train.")


## 7. Transformer Model

After the DLinear shape check works, use the repo's existing `iTransformerForForecasting`. It receives the same `[batch_size, input_len, num_features]` input and returns `[batch_size, 6, num_features]`. In this split-horizon hybrid, the Transformer is responsible for forecast steps 7-12.


In [ ]:
#transfoemr model build it
transformer_model_config = iTransformerConfig(
    input_len=INPUT_LEN,
    output_len=SPLIT_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    dropout=0.1,
    use_revin=True,
)

transformer_cfg = BasicTSForecastingConfig(
    model=iTransformerForForecasting,
    model_config=transformer_model_config,
    taskflow=SplitHorizonForecastingTaskFlow(slice(SPLIT_OUTPUT_LEN, FULL_OUTPUT_LEN)),
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/iTransformerForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_7_12",
    **SHARED_CONFIG,
)

# Standalone Transformer trained to forecast all 12 steps.
transformer_full_model_config = iTransformerConfig(
    input_len=INPUT_LEN,
    output_len=FULL_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    dropout=0.1,
    use_revin=True,
)

transformer_full_cfg = BasicTSForecastingConfig(
    model=iTransformerForForecasting,
    model_config=transformer_full_model_config,
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/iTransformerForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **FULL_12_STEP_CONFIG,
)

transformer_cfg, transformer_full_cfg


## 8. Transformer Shape Test

Run this after the DLinear section works. It uses the same shared data pipeline and checks that the Transformer predicts only its 6-step split-horizon target.


In [ ]:
#check the shapes
transformer_batch, transformer_prediction = preview_shapes(transformer_cfg, "Transformer")


## 9. Train the Transformer

Use the same dataset, scaler, preprocessing, and `input_len` as the DLinear. The shared dataset still contains the full 12-step target window, but the Transformer taskflow slices that target to steps 7-12.


In [ ]:
#train trasnfoemr
RUN_TRANSFORMER_TRAINING = False
RUN_TRANSFORMER_12_STEP_TRAINING = False

if RUN_TRANSFORMER_TRAINING:
    transformer_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "iTransformerForForecasting",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_7_12",
    )
    print("Training split Transformer in:", transformer_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(transformer_cfg)
else:
    print("Transformer split-horizon training skipped. Set RUN_TRANSFORMER_TRAINING = True after the DLinear works.")

if RUN_TRANSFORMER_12_STEP_TRAINING:
    transformer_full_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "iTransformerForForecasting",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    )
    print("Training 12-step Transformer in:", transformer_full_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(transformer_full_cfg)
else:
    print("Transformer 12-step training skipped. Set RUN_TRANSFORMER_12_STEP_TRAINING = True to train.")


## 13. Selector-Based Hybrid: Hard and Soft Weighted Selection Without Test Leakage

In this section, we move beyond the fixed split-horizon hybrid (DLinear steps 1-6, Transformer steps 7-12) to intelligent selection mechanisms:

1. **Hard Selector**: For each forecast step and feature, choose the model with the lowest validation MAE.
2. **Soft Weighted Selector**: For each forecast step and feature, blend predictions using weights derived from inverse validation MAE.

The important rule: validation data is used to learn the selector, and test data is used only for final evaluation.

In [ ]:
# Generate full 12-step validation and test predictions for selector comparison.
# The selector is learned on validation predictions only, then applied to test predictions.

from types import SimpleNamespace

print("Loading full 12-step model predictions for VAL and TEST...")


def latest_best_checkpoint(cfg):
    metric_name = cfg.target_metric.replace("/", "_")
    checkpoint_name = f"{cfg.model.__name__}_best_val_{metric_name}.pt"
    checkpoint_files = sorted(
        Path(cfg.ckpt_save_dir).rglob(checkpoint_name),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not checkpoint_files:
        raise FileNotFoundError(
            f"No {checkpoint_name} found under {cfg.ckpt_save_dir}. Train the model first."
        )
    return checkpoint_files[0]


def predict_full_model_on_mode(cfg, mode):
    train_dataset = Builder._build_dataset(cfg, BasicTSMode.TRAIN)
    eval_dataset = Builder._build_dataset(cfg, mode)
    eval_loader = DataLoader(eval_dataset, batch_size=cfg.batch_size, shuffle=False)

    scaler = Builder._build_scaler(cfg) if cfg.scaler is not None else None
    if scaler is not None:
        scaler.fit(train_dataset.data)

    model = cfg.model(cfg.model_config)
    checkpoint_path = latest_best_checkpoint(cfg)
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    runner = SimpleNamespace(cfg=cfg, scaler=scaler)
    predictions = []
    targets = []

    with torch.no_grad():
        for raw_batch in eval_loader:
            batch = _float_batch(raw_batch)
            batch = cfg.taskflow.preprocess(runner, batch)
            prediction = model(batch["inputs"])
            if isinstance(prediction, dict):
                prediction = prediction["prediction"]
            predictions.append(prediction.cpu().numpy())
            targets.append(batch["targets"].cpu().numpy())

    return np.concatenate(predictions, axis=0), np.concatenate(targets, axis=0), checkpoint_path


dlinear_full_val_pred, dlinear_full_val_targets, dlinear_full_ckpt = predict_full_model_on_mode(
    dlinear_full_cfg,
    BasicTSMode.VAL,
)
transformer_full_val_pred, transformer_full_val_targets, transformer_full_ckpt = predict_full_model_on_mode(
    transformer_full_cfg,
    BasicTSMode.VAL,
)

dlinear_full_test_pred, dlinear_full_test_targets, _ = predict_full_model_on_mode(
    dlinear_full_cfg,
    BasicTSMode.TEST,
)
transformer_full_test_pred, transformer_full_test_targets, _ = predict_full_model_on_mode(
    transformer_full_cfg,
    BasicTSMode.TEST,
)

print(f"DLinear best checkpoint: {dlinear_full_ckpt}")
print(f"Transformer best checkpoint: {transformer_full_ckpt}")
print(f"DLinear VAL pred shape: {dlinear_full_val_pred.shape}")
print(f"Transformer VAL pred shape: {transformer_full_val_pred.shape}")
print(f"DLinear TEST pred shape: {dlinear_full_test_pred.shape}")
print(f"Transformer TEST pred shape: {transformer_full_test_pred.shape}")

assert dlinear_full_val_pred.shape == dlinear_full_val_targets.shape
assert transformer_full_val_pred.shape == transformer_full_val_targets.shape
assert dlinear_full_test_pred.shape == dlinear_full_test_targets.shape
assert transformer_full_test_pred.shape == transformer_full_test_targets.shape
assert dlinear_full_val_pred.shape[1:] == (FULL_OUTPUT_LEN, NUM_FEATURES)
assert dlinear_full_test_pred.shape[1:] == (FULL_OUTPUT_LEN, NUM_FEATURES)

In [ ]:
# Hard Selector: Learn per-step/per-feature choices on VAL, then apply to TEST.

print("\n=== HARD SELECTOR (Validation-Tuned Step-wise Best Model) ===")

dlinear_val_per_step_mae = np.mean(
    np.abs(dlinear_full_val_pred - dlinear_full_val_targets),
    axis=0,
)

transformer_val_per_step_mae = np.mean(
    np.abs(transformer_full_val_pred - transformer_full_val_targets),
    axis=0,
)

print(f"DLinear validation per-step MAE shape: {dlinear_val_per_step_mae.shape}")
print(f"Transformer validation per-step MAE shape: {transformer_val_per_step_mae.shape}")

# hard_selector_mask[step, feature] = 0 for DLinear, 1 for Transformer.
# This is learned only from validation errors.
#make this more generalizable
hard_selector_mask = (
    transformer_val_per_step_mae < dlinear_val_per_step_mae
).astype(int)

hard_selected_pred = np.where(
    hard_selector_mask[np.newaxis, :, :],
    transformer_full_test_pred,
    dlinear_full_test_pred,
)

hard_selected_targets = dlinear_full_test_targets

print(f"Hard selector mask shape: {hard_selector_mask.shape}")
print(f"Hard-selected TEST prediction shape: {hard_selected_pred.shape}")
print(f"Hard-selected TEST targets shape: {hard_selected_targets.shape}")

assert np.allclose(dlinear_full_test_targets, transformer_full_test_targets)
assert hard_selected_pred.shape == hard_selected_targets.shape

print("\nHard selector learned from VAL: Model choices per step (% Transformer):")
for step in range(FULL_OUTPUT_LEN):
    pct_transformer = np.mean(hard_selector_mask[step, :]) * 100
    print(f"  Step {step+1}: {pct_transformer:.1f}% Transformer, {100-pct_transformer:.1f}% DLinear")

In [ ]:
# Soft Weighted Selector: Learn inverse-MAE weights on VAL, then apply to TEST.

print("\n=== SOFT WEIGHTED SELECTOR (Validation-Tuned Weighted Average) ===")

epsilon = 1e-6

dlinear_weights = 1.0 / (dlinear_val_per_step_mae + epsilon)
transformer_weights = 1.0 / (transformer_val_per_step_mae + epsilon)

total_weights = dlinear_weights + transformer_weights
dlinear_weights_normalized = dlinear_weights / total_weights
transformer_weights_normalized = transformer_weights / total_weights

weighted_pred = (
    dlinear_weights_normalized[np.newaxis, :, :] * dlinear_full_test_pred
    + transformer_weights_normalized[np.newaxis, :, :] * transformer_full_test_pred
)

weighted_targets = dlinear_full_test_targets

print(f"DLinear validation-derived weights shape: {dlinear_weights_normalized.shape}")
print(f"Transformer validation-derived weights shape: {transformer_weights_normalized.shape}")
print(f"Weighted TEST prediction shape: {weighted_pred.shape}")
print(f"Weighted TEST targets shape: {weighted_targets.shape}")

assert weighted_pred.shape == weighted_targets.shape

print("\nSoft weights learned from VAL, averaged across features:")
print(f"{'Step':<6} {'DLinear Weight':<12} {'Transformer Weight':<18}")
print("-" * 36)

for step in range(FULL_OUTPUT_LEN):
    dlinear_step_weight = np.mean(dlinear_weights_normalized[step, :])
    transformer_step_weight = np.mean(transformer_weights_normalized[step, :])
    print(f"{step+1:<6} {dlinear_step_weight:<12.4f} {transformer_step_weight:<18.4f}")

In [ ]:
# Comprehensive metrics table.
# The selector hybrids were tuned on VAL and are evaluated here on TEST only.

print("\n=== COMPREHENSIVE TEST METRICS COMPARISON ===\n")

all_comparison = {
    "Transformer split steps 7-12": compute_metrics(transformer_test_pred, transformer_test_targets),
    "Fixed split hybrid 1-12": compute_metrics(hybrid_test_pred, hybrid_test_targets),
    "DLinear full steps 1-12": compute_metrics(dlinear_full_test_pred, dlinear_full_test_targets),
    "Transformer full steps 1-12": compute_metrics(transformer_full_test_pred, transformer_full_test_targets),
    "Hard selector hybrid 1-12": compute_metrics(hard_selected_pred, hard_selected_targets),
    "Soft weighted hybrid 1-12": compute_metrics(weighted_pred, weighted_targets),
}

print(f"{'Model':<35} {'MAE':>12} {'MSE':>12}")
print("-" * 60)
for model_name in [
    "Transformer split steps 7-12",
    "Fixed split hybrid 1-12",
    "DLinear full steps 1-12",
    "Transformer full steps 1-12",
    "Hard selector hybrid 1-12",
    "Soft weighted hybrid 1-12",
]:
    metrics = all_comparison[model_name]
    mae = metrics["MAE"]
    mse = metrics["MSE"]
    print(f"{model_name:<35} {mae:>12.6f} {mse:>12.6f}")

best_mae_model = min(all_comparison.items(), key=lambda x: x[1]["MAE"])
best_mse_model = min(all_comparison.items(), key=lambda x: x[1]["MSE"])

print("\n" + "=" * 60)
print(f"Best TEST MAE: {best_mae_model[0]:<30} {best_mae_model[1]['MAE']:.6f}")
print(f"Best TEST MSE: {best_mse_model[0]:<30} {best_mse_model[1]['MSE']:.6f}")

In [ ]:
# Save selector hybrid predictions to the mixed run folder in checkpoints/.

print("\n=== SAVING LEAKAGE-SAFE SELECTOR PREDICTIONS ===\n")

mixed_output_dir = Path("checkpoints") / RUN_TAG
mixed_output_dir.mkdir(parents=True, exist_ok=True)

hard_selector_save_path = mixed_output_dir / "hard_selector_val_tuned_hybrid_dlinear_transformer_ETTh1_96_12_prediction.npy"
np.save(hard_selector_save_path, hard_selected_pred)
print(f"Saved hard selector predictions: {hard_selector_save_path}")
print(f"  Shape: {hard_selected_pred.shape}")

weighted_selector_save_path = mixed_output_dir / "soft_weighted_val_tuned_hybrid_dlinear_transformer_ETTh1_96_12_prediction.npy"
np.save(weighted_selector_save_path, weighted_pred)
print(f"Saved soft weighted predictions: {weighted_selector_save_path}")
print(f"  Shape: {weighted_pred.shape}")

weights_info = {
    "selector_fit_split": "validation",
    "final_evaluation_split": "test",
    "dlinear_weights": dlinear_weights_normalized.tolist(),
    "transformer_weights": transformer_weights_normalized.tolist(),
    "hard_selector_mask": hard_selector_mask.tolist(),
    "description": "Hard selector mask: 0=DLinear, 1=Transformer. Soft weights are normalized inverse validation MAE per step and feature.",
}
weights_save_path = mixed_output_dir / "selector_metadata_val_tuned_ETTh1_96_12.json"
with weights_save_path.open("w", encoding="utf-8") as f:
    json.dump(weights_info, f, indent=2)
print(f"Saved selector metadata: {weights_save_path}")

hybrid_metrics_save_path = mixed_output_dir / "hybrid_metrics_ETTh1_96_12.json"
with hybrid_metrics_save_path.open("w", encoding="utf-8") as f:
    json.dump(all_comparison, f, indent=2)
print(f"Saved hybrid metrics: {hybrid_metrics_save_path}")

print("All leakage-safe selector predictions and metadata saved successfully.")